In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import re
import dateparser
from datetime import datetime

In [ ]:
url = "LINK" #add link

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
}

In [ ]:
# number of pages

response = requests.get(url, headers=headers, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")

pages = soup.select('ul.page-pagination li a.page-numbers:not(.next)')


page_numbers = [int(a.text) for a in pages if a.text.isdigit()]
total_pages = max(page_numbers) if page_numbers else 1

print("Total number of pages:", total_pages )

In [ ]:
# list of the links of all tenders

list_links = []

for page in range(1, total_pages + 1):
    if page == 1:
        base_url = url
    else:
        base_url = f"{url}page/{page}/"
    
    response = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    h4= soup.find_all("h4", class_="entry-title")
    for element in h4:
        link = element.find("a")["href"]
        list_links.append(link)

print(list_links)

In [ ]:
# detect the dates

months_fr = [
    "janvier", "février", "mars", "avril", "mai", "juin",
    "juillet", "août", "septembre", "octobre", "novembre", "décembre"
]

months_regex = "|".join(months_fr)

pattern = rf"""\b(
    \d{{1,2}}[/-]\d{{1,2}}[/-]\d{{4}} |        
    \d{{1,2}}\s+(?:{months_regex})\s+\d{{4}} 
)\b"""

In [ ]:
# list of links of non-expired tenders

list_available_links=[]

for l in list_links:
    response = requests.get(l, headers=headers)
    soup= BeautifulSoup(response.text, "html.parser")
    entry = soup.find("div", class_="entry-summary")
    if entry:
        text = entry.get_text(separator=" ", strip=True)

    dates = re.findall(pattern, text, re.IGNORECASE | re.VERBOSE)
    if dates:
        date = (dates[0].strip())
        expiration_date = dateparser.parse(date, languages=['fr']).date()

        if expiration_date and expiration_date >= datetime.now().date() :
            list_available_links.append(l)
    else:
        continue

print(list_available_links)

In [ ]:
# scraping

tenders = []
url = "https://amrtp.ml/category/appels-doffres/"

for page in range(1, total_pages + 1):
    if page == 1:
        base_url = url
    else:
        base_url = f"{url}page/{page}/"

    response = requests.get(base_url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    
    blocks_list = soup.find_all("div", class_="inner-post")
    for tag in blocks_list:
        h4 = tag.find("h4", class_="entry-title")
        link= h4.find("a")["href"]

        if  link in list_available_links: 
            publish_date = tag.find("span", class_="posted-on").text
    
            subject = tag.find("div", class_="entry-summary").text
            title = f"{h4.text}\n{subject}"

            response = requests.get(link, headers=headers)
            soup= BeautifulSoup(response.text, "html.parser")
            entry = soup.find("div", class_="entry-summary")
            if entry:
                text = entry.get_text(separator=" ", strip=True)
            dates = re.findall(pattern, text, re.IGNORECASE | re.VERBOSE)
            if dates:
                date = (dates[0].strip())
                expiration_date = dateparser.parse(date, languages=['fr']).date()
        

            tenders.append({
                "title": title,
                "publication_date": publish_date,
                "expiration_date": expiration_date,
                "link": link,
                "country": "Mali"
            })

print(tenders)

In [ ]:
def clean_excel_string(s):
    if isinstance(s, str):
        return re.sub(r'[\x00-\x1F\x7F-\x9F]', '', s)
    return s

df = pd.DataFrame(tenders)
df = df.applymap(clean_excel_string)

df.to_excel("../2_scraped_files/Mali.xlsx", index=False) # modify